# nb_03a — Gold : dimensions (SCD1 + deux variantes de SCD Type 2)

**Module 3 (dimensions).**

| Dimension | Technique | Pourquoi |
|---|---|---|
| `dim_date` | générée | calendrier des dates |
| `dim_cost_center` | SCD1 (overwrite) | les centres de coûts sont stables ; porte la région RLS |
| `dim_pay_band` | **SCD2 à partir d'un flux d'historique** | les grilles sont réévaluées chaque année ; l'historique complet est disponible |
| `dim_worker` | **SCD2 via MERGE de snapshots périodiques** | les actions de gestion des effectifs modifient les enregistrements des employés |

Chaque dimension reçoit une **clé de substitution** entière afin que la table de faits puisse retrouver la
*version en vigueur à la date de l'événement* (nb_03b).

## Imports et schéma Gold

**Résumé.** Charge les utilitaires PySpark et Delta, puis crée le schéma `gold` dans lequel les dimensions sont écrites.

<details>
<summary>Détails ligne par ligne</summary>

- `from pyspark.sql import functions as F, Window as W` — expressions et fonctions de fenêtrage.
- `from delta.tables import DeltaTable` — l'API utilisée ensuite pour le MERGE de `dim_worker`.
- `CREATE SCHEMA IF NOT EXISTS gold` — vérifie que le schéma Gold existe.

</details>

In [ ]:
from pyspark.sql import functions as F, Window as W
from delta.tables import DeltaTable
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

## 1. `dim_date` — Créer la table des dates

**Résumé.** Génère un calendrier quotidien continu avec les clés et attributs (année, année fiscale, trimestre, mois) qui alimentent l'intelligence temporelle du modèle.

<details>
<summary>Détails ligne par ligne</summary>

- `sequence(to_date('2021-01-01'), to_date('2025-12-31'), interval 1 day)` dans `explode(...)` — une ligne par jour sur toute la période du laboratoire.
- `date_key` — la clé de substitution entière au format `yyyyMMdd`.
- `year`, `fiscal_year` (début en avril), `quarter`, `month`, `month_name` — attributs calendaires dérivés.
- `write ... saveAsTable("gold.dim_date")` et `print` — enregistrent la table et affichent le nombre de lignes.

</details>

In [ ]:
dates = (spark.sql("""
  SELECT explode(sequence(to_date('2021-01-01'), to_date('2025-12-31'),
                          interval 1 day)) AS date""")
  .withColumn("date_key", F.date_format("date","yyyyMMdd").cast("int"))
  .withColumn("year", F.year("date"))
  .withColumn("fiscal_year",
      F.when(F.month("date")>=4, F.year("date")).otherwise(F.year("date")-1))
  .withColumn("quarter", F.concat(F.year("date"), F.lit("-Q"), F.quarter("date")))
  .withColumn("month", F.date_format("date","yyyy-MM"))
  .withColumn("month_name", F.date_format("date","MMMM")))

(dates.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold.dim_date"))

print(f"dim_date: {dates.count():,} rows")

## 2. `dim_cost_center` — SCD Type 1

**Résumé.** Remplace l'état actuel des centres de coûts et ajoute une clé de substitution ; porte `hr_region`.

<details>
<summary>Détails ligne par ligne</summary>

- `cc = spark.table("bronze.cost_centers")` — la table maître source.
- `F.row_number().over(W.orderBy("cost_center_id"))` — une clé entière stable `cost_center_key`.
- `.select(...)` — conserve la clé ainsi que les colonnes descriptives, notamment `hr_region`.
- `write ... saveAsTable("gold.dim_cost_center")` — remplace la table (Type 1, sans historique).

</details>

In [ ]:
cc = spark.table("bronze.cost_centers")

dim_cost_center = (cc.withColumn("cost_center_key", F.row_number().over(W.orderBy("cost_center_id")))
    .select("cost_center_key","cost_center_id","cost_center_name","branch",
            "hr_region","business_line"))

(dim_cost_center.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold.dim_cost_center"))

print("dim_cost_center:", dim_cost_center.count())

## 3. `dim_pay_band` — SCD Type 2 à partir d'un flux d'historique

**Résumé.** Transforme l'historique des tranches par date d'effet en périodes de validité : l'`effective_to` de chaque version est le jour précédant la version suivante, et la dernière version est marquée comme étant la version courante.

<details>
<summary>Détails ligne par ligne</summary>

- `W.partitionBy("classification_group","classification_level").orderBy("band_effective_date")` — ordonne les versions au sein de chaque cellule de grille.
- `effective_from = band_effective_date` et `_next = F.lead(...)` — les dates d'effet actuelle et suivante.
- `effective_to = when(_next is null, 9999-12-31).otherwise(date_sub(_next, 1))` — clôture chaque version la veille de la suivante.
- `is_current = _next.isNull()` — la dernière version sans date de fin.
- `pay_band_key` via `row_number()` — la clé de substitution que la table de faits résout à la date de l'événement.
- `write ... saveAsTable("gold.dim_pay_band")` ainsi que l'aperçu `show(...)` de l'historique PA / niveau 3.

</details>

In [ ]:
hist = spark.table("bronze.pay_bands")

w = W.partitionBy("classification_group","classification_level").orderBy("band_effective_date")

scd2 = (hist
    .withColumn("effective_from", F.col("band_effective_date"))
    .withColumn("_next", F.lead("band_effective_date").over(w))
    .withColumn("effective_to",
        F.when(F.col("_next").isNull(), F.to_date(F.lit("9999-12-31")))
         .otherwise(F.date_sub("_next",1)))
    .withColumn("is_current", F.col("_next").isNull())
    .drop("_next","band_effective_date","_ingested_at"))

dim_pay_band = (scd2.withColumn("pay_band_key",
        F.row_number().over(W.orderBy("classification_group","classification_level","effective_from")))
    .select("pay_band_key","classification_group","classification_level",
            "band_min","band_mid","band_max","effective_from","effective_to","is_current"))

(dim_pay_band.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold.dim_pay_band"))

print(f"dim_pay_band: {dim_pay_band.count():,} versioned rows")

dim_pay_band.filter("classification_group='PA' AND classification_level=3") \
    .orderBy("effective_from") \
    .select("band_mid","effective_from","effective_to","is_current").show(truncate=False)

## 4. `dim_worker` — SCD Type 2 via MERGE et insertion des nouvelles versions



Le processus canonique **close-then-insert** suit `classification_group`,

`classification_level`, `directorate`, `employment_type` via un `row_hash`.

Il se déroule en deux opérations : un MERGE ferme les versions courantes modifiées,

puis un `append` insère les nouvelles versions.



1. **Chargement initial** — chaque employé devient la version 1 (`is_current=true`,

   `effective_to=9999-12-31`).

2. **Deuxième snapshot** (`workers_delta`, effectif au 2023-07-01) — les lignes modifiées sont

   **clôturées**, puis la nouvelle version est **insérée** ; les nouveaux employés sont insérés directement.


<details>
<summary>Détails ligne par ligne</summary>

- `def hash_cols(*cols)` — applique `sha2` aux colonnes suivies concaténées (les valeurs nulles sont remplacées par une chaîne vide) afin de produire un `row_hash` de détection des changements.
- `TRACK = [...]` — les attributs dont les changements ouvrent une nouvelle version.
- `snap1 = spark.table("bronze.workers")` — le premier snapshot.
- `init` — convertit `classification_level`, calcule `row_hash`, définit `effective_from` à partir de la date du snapshot, `effective_to = 9999-12-31` et `is_current = true`.
- `write ... saveAsTable("gold.dim_worker")` — écrit la version 1 pour chaque employé.

</details>

In [ ]:
def hash_cols(*cols):
    return F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c),F.lit("")) for c in cols]),256)
TRACK = ["classification_group","classification_level","directorate","employment_type"]

snap1 = spark.table("bronze.workers")

init = (snap1
    .withColumn("classification_level", F.col("classification_level").cast("int"))
    .withColumn("row_hash", hash_cols(*TRACK))
    .withColumn("effective_from", F.to_date("snapshot_date"))
    .withColumn("effective_to", F.to_date(F.lit("9999-12-31")))
    .withColumn("is_current", F.lit(True))
    .select("employee_id","full_name","home_cost_center_id",*TRACK,"row_hash",
            "effective_from","effective_to","is_current"))

(init.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold.dim_worker"))

print(f"dim_worker initial: {init.count():,} rows")

## `dim_worker` — appliquer le deuxième snapshot (MERGE close-then-insert)

**Résumé.** Applique le snapshot ultérieur `workers_delta` : le MERGE clôture les lignes courantes modifiées, puis les employés modifiés et les nouveaux employés sont insérés comme nouvelles versions courantes.

<details>
<summary>Détails ligne par ligne</summary>

- `snap2` — le deuxième snapshot avec `row_hash` et `effective_from` calculés de la même manière.
- `DeltaTable.forName(spark, "gold.dim_worker")` — la cible du MERGE.
- Étape 1, MERGE sur `employee_id` + `is_current = true`, `whenMatchedUpdate(condition="t.row_hash <> s.row_hash", ...)` — clôture uniquement les lignes dont les attributs suivis ont changé en définissant `is_current=false` et `effective_to` au jour précédant le nouveau snapshot.
- Étape 2 : `current` = les lignes toujours courantes ; `changed_or_new` effectue une jointure gauche du snapshot avec celles-ci et conserve les lignes nouvelles (`c.employee_id IS NULL`) ou modifiées (`row_hash` différent), avec `effective_to=9999-12-31` et `is_current=true`.
- `changed_or_new.write ... mode("append")` — insère les nouvelles versions.
- Le `SELECT is_current, count(*) ...` final affiche la répartition entre les versions courantes et historiques.

</details>

In [ ]:
snap2 = (spark.table("bronze.workers_delta")

    .withColumn("classification_level", F.col("classification_level").cast("int"))

    .withColumn("row_hash", hash_cols(*TRACK))

    .withColumn("effective_from", F.to_date("snapshot_date")))



tgt = DeltaTable.forName(spark, "gold.dim_worker")

# Étape 1 : FERMER les versions courantes modifiées



(tgt.alias("t").merge(snap2.alias("s"),

        "t.employee_id = s.employee_id AND t.is_current = true")

   .whenMatchedUpdate(condition="t.row_hash <> s.row_hash",

        set={"is_current": F.lit(False),

             "effective_to": F.expr("date_sub(s.effective_from, 1)")})

   .execute())



# Étape 2 : INSÉRER les nouvelles versions (modifiées) et les nouveaux employés

current = spark.table("gold.dim_worker").filter("is_current = true")



changed_or_new = (snap2.alias("s")

    .join(current.alias("c"), "employee_id", "left")

    .where("c.employee_id IS NULL OR c.row_hash <> s.row_hash")

    .select("s.employee_id","s.full_name","s.home_cost_center_id",

            *[f"s.{x}" for x in TRACK],"s.row_hash","s.effective_from",

            F.to_date(F.lit("9999-12-31")).alias("effective_to"),

            F.lit(True).alias("is_current")))



changed_or_new.write.format("delta").mode("append").saveAsTable("gold.dim_worker")



spark.sql("""SELECT is_current, count(*) rows FROM gold.dim_worker

             GROUP BY is_current ORDER BY is_current""").show()


## `dim_worker` — attribuer les clés de substitution

**Résumé.** Ajoute une clé entière stable `worker_key` à toutes les versions des employés afin que la table de faits puisse retrouver la version exacte en vigueur à la date d'un événement.

<details>
<summary>Détails ligne par ligne</summary>

- `F.row_number().over(W.orderBy("employee_id","effective_from"))` — une clé de substitution déterministe par version.
- `write ... mode("overwrite") ... saveAsTable("gold.dim_worker")` — réécrit la dimension avec les clés.
- Le `SELECT ...` final affiche un aperçu des employés qui possèdent plusieurs versions.

</details>

In [ ]:
dim_w = (spark.table("gold.dim_worker")
    .withColumn("worker_key", F.row_number().over(W.orderBy("employee_id","effective_from"))))

(dim_w.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold.dim_worker"))

spark.sql("""
  SELECT employee_id, classification_group, classification_level, directorate,
         effective_from, effective_to, is_current
  FROM gold.dim_worker
  WHERE employee_id IN (SELECT employee_id FROM gold.dim_worker
                        GROUP BY employee_id HAVING count(*)>1)
  ORDER BY employee_id, effective_from""").show(10, truncate=False)